# Carbon Tree Species — Image Classification Pipeline (EfficientNetB0)

End-to-end pipeline: read species + taxon IDs from your xlsx -> live-verify both external APIs ->
download research-grade photos from iNaturalist -> validate/clean -> split -> augment (capped) ->
train EfficientNetB0 with regularization tuned to avoid overfitting -> evaluate on a held-out test
set -> export only the trained model artifacts (best checkpoint, INT8 + float32 TFLite, confusion
matrix, training curves, logs).

**Design notes worth knowing before you run this:**
- The xlsx's `gbif_taxon_key` column is not trusted for verification — GBIF taxon keys are resolved
  live by **name match** (`species/match?name=...`) instead, since static keys can silently point at
  the wrong species and there'd be no way to tell from the data alone. iNaturalist's `taxon_id` values
  are also live-verified the same way, against the taxon's actual returned name.
- `dbh_cm`, `height_m`, `co2_kg_yr` in the xlsx are **species-level averages**, not per-photo
  measurements — every photo of a species shares the same number. So this pipeline does not bolt on
  a fake "height/quality from pixels" head: training one against a constant value per class would just
  memorize a lookup table, not read anything from the image. Instead, the model predicts species (the
  one thing the photos genuinely support), and a small `species_lookup.json` maps that prediction to
  its known height/DBH/CO2 attributes. Real per-image height/quality prediction would need labels that
  actually vary photo-to-photo (e.g. a human measuring/tagging each one).
- Only model artifacts are zipped for download — no raw/train/val/test image dump.
- TensorFlow Lite supports float32, float16, and int8 quantization. It has no 4-bit weight-quantization
  path for a standard CNN like this — 4-bit schemes (GPTQ/AWQ/GGUF) are an LLM-specific toolchain with
  no equivalent runtime for a CNN like this. **INT8** is the lowest precision that's actually real and
  runnable here, so that's what gets exported, alongside a float32 version for accuracy-sensitive use.

Run top-to-bottom in Google Colab. Upload `carbon_pilot_dataset.xlsx` when prompted in Cell 2.

In [1]:
# -- Cell 1: Install dependencies --------------------------------------------
!pip install -q opencv-python-headless requests Pillow tqdm scikit-learn openpyxl

import os, json, time, math, random, shutil, csv
import requests
from PIL import Image, ImageEnhance
import numpy as np
import cv2
from tqdm import tqdm
from pathlib import Path
import openpyxl
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print('All packages ready.')

TensorFlow : 2.20.0
NumPy      : 2.0.2
All packages ready.


In [2]:
# -- Cell 2: Upload xlsx, parse species, and LIVE-VERIFY both APIs -----------
from google.colab import files as colab_files

print('Upload carbon_pilot_dataset.xlsx when the file picker appears ...')
uploaded  = colab_files.upload()
XLSX_PATH = next(iter(uploaded))
print(f'Received: {XLSX_PATH}')

wb = openpyxl.load_workbook(XLSX_PATH, read_only=True, data_only=True)
print('Sheets found:', wb.sheetnames)

def sheet_to_dicts(ws):
    rows = list(ws.iter_rows(values_only=True))
    header_idx = None
    for i, row in enumerate(rows):
        non_empty = [c for c in row if c is not None and str(c).strip()]
        if len(non_empty) >= 3:
            header_idx = i
            break
    if header_idx is None:
        return []
    headers = [str(c).strip() if c is not None else f'col{j}'
               for j, c in enumerate(rows[header_idx])]
    result = []
    for row in rows[header_idx + 1:]:
        if all(v is None for v in row):
            continue
        d = dict(zip(headers, row))
        first_val = str(d.get(headers[0], ''))
        if first_val.startswith('http') or 'API:' in first_val:
            continue
        result.append(d)
    return result

ref_rows  = sheet_to_dicts(wb[wb.sheetnames[0]])   # species_reference
inat_rows = sheet_to_dicts(wb[wb.sheetnames[2]])   # inat_download_guide

inat_lookup = {}
for r in inat_rows:
    code = str(r.get('species_code', '')).strip()
    if not code or len(code) > 6:
        continue
    try:
        inat_lookup[code] = dict(
            inat_id  = int(float(str(r['inat_taxon_id']).strip())),
            gbif_key = int(float(str(r['gbif_taxon_key']).strip())),  # kept for reference only, NOT trusted
        )
    except (TypeError, ValueError, KeyError) as e:
        print(f'  Skipping inat row for {code}: {e}')

SPECIES = []
for r in ref_rows:
    code = str(r.get('species_code', '')).strip()
    if not code or len(code) > 5:
        continue
    inat_info = inat_lookup.get(code, {})
    if not inat_info:
        print(f'  WARNING: no iNat/GBIF info for {code}, skipping')
        continue
    SPECIES.append(dict(
        code        = code,
        common      = str(r.get('common_name', code)),
        scientific  = str(r.get('scientific_name', '')),
        family      = str(r.get('family', '')),
        co2_class   = str(r.get('co2_class', 'MEDIUM')),
        co2_kg_yr   = float(r.get('co2_kg_yr_avg') or 0),
        dbh_cm      = float(r.get('dbh_typical_cm') or 0),
        height_m    = float(r.get('height_typical_m') or 0),
        biome       = str(r.get('biome', 'Tropical')),
        inat_id     = inat_info['inat_id'],
        gbif_key_xlsx = inat_info['gbif_key'],   # xlsx's key, kept only for the verification report below
    ))
wb.close()
if not SPECIES:
    raise RuntimeError('No species loaded - check your xlsx sheet structure.')
print(f'\nLoaded {len(SPECIES)} species from {XLSX_PATH}')

# -- Live API verification ----------------------------------------------------
# iNaturalist: confirm taxon_id resolves and returns the expected scientific name.
# GBIF: resolve by NAME MATCH (species/match?name=...) instead of trusting the xlsx's
#       gbif_taxon_key column -- that key was found to point at unrelated species during
#       manual testing, so we don't rely on it for verification at all.
INAT_TAXON_URL = 'https://api.inaturalist.org/v1/taxa/{id}'
GBIF_MATCH_URL = 'https://api.gbif.org/v1/species/match'

print('\nVerifying APIs against the xlsx data ...')
api_report = []
for sp in SPECIES:
    row = dict(code=sp['code'], scientific=sp['scientific'])

    # --- iNaturalist check ---
    try:
        r = requests.get(INAT_TAXON_URL.format(id=sp['inat_id']), timeout=15)
        r.raise_for_status()
        results = r.json().get('results', [])
        inat_name = results[0]['name'] if results else None
        row['inat_ok'] = (inat_name is not None and
                           inat_name.split()[:2] == sp['scientific'].split()[:2])
        row['inat_returned'] = inat_name
    except Exception as exc:
        row['inat_ok'] = False
        row['inat_returned'] = f'ERROR: {exc}'

    # --- GBIF check (name match, NOT the xlsx key) ---
    try:
        r = requests.get(GBIF_MATCH_URL, params={'name': sp['scientific']}, timeout=15)
        r.raise_for_status()
        data = r.json()
        gbif_name = data.get('canonicalName') or data.get('scientificName')
        row['gbif_ok'] = (gbif_name is not None and
                           gbif_name.split()[:2] == sp['scientific'].split()[:2])
        row['gbif_resolved_key'] = data.get('usageKey')
        row['gbif_returned'] = gbif_name
    except Exception as exc:
        row['gbif_ok'] = False
        row['gbif_resolved_key'] = None
        row['gbif_returned'] = f'ERROR: {exc}'

    # Replace the (possibly wrong) xlsx key with the freshly-resolved one for downstream use.
    sp['gbif_key'] = row['gbif_resolved_key'] or sp['gbif_key_xlsx']

    api_report.append(row)
    flag_i = 'OK' if row['inat_ok'] else 'FAIL'
    flag_g = 'OK' if row['gbif_ok'] else 'FAIL'
    print(f"  {sp['code']:<4} iNat[{flag_i}]={row['inat_returned']!s:<28} "
          f"GBIF[{flag_g}]={row['gbif_returned']} (resolved key={row['gbif_resolved_key']})")
    time.sleep(0.3)

n_inat_ok = sum(1 for r in api_report if r['inat_ok'])
n_gbif_ok = sum(1 for r in api_report if r['gbif_ok'])
print(f'\niNaturalist taxon IDs verified: {n_inat_ok}/{len(SPECIES)}')
print(f'GBIF names resolved by match : {n_gbif_ok}/{len(SPECIES)}')
print('Note: xlsx gbif_taxon_key column is NOT used for training/verification -- '
      'it was found unreliable. Resolved keys above are used instead.')

Upload carbon_pilot_dataset.xlsx when the file picker appears ...


Saving carbon_pilot_dataset.xlsx to carbon_pilot_dataset.xlsx
Received: carbon_pilot_dataset.xlsx
Sheets found: ['species_reference', 'labels_template', 'inat_download_guide']

Loaded 8 species from carbon_pilot_dataset.xlsx

Verifying APIs against the xlsx data ...
  TEK  iNat[OK]=Tectona grandis              GBIF[OK]=Tectona grandis (resolved key=2925649)
  NEM  iNat[OK]=Azadirachta indica           GBIF[OK]=Azadirachta indica (resolved key=3190474)
  MNG  iNat[OK]=Mangifera indica             GBIF[OK]=Mangifera indica (resolved key=3190638)
  PEP  iNat[OK]=Ficus religiosa              GBIF[OK]=Ficus religiosa (resolved key=5361935)
  BAM  iNat[OK]=Bambusa vulgaris             GBIF[OK]=Bambusa vulgaris (resolved key=7661971)
  MGR  iNat[OK]=Rhizophora mangle            GBIF[OK]=Rhizophora mangle (resolved key=3086528)
  RSW  iNat[OK]=Dalbergia latifolia          GBIF[OK]=Dalbergia latifolia (resolved key=2968662)
  MOR  iNat[OK]=Moringa oleifera             GBIF[OK]=Moringa oleifera 

In [3]:
# -- Cell 3: Folder layout and pipeline constants -----------------------------
BASE_DIR   = Path('/content/tree_dataset')
BASE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR    = BASE_DIR / 'raw'
TRAIN_DIR  = BASE_DIR / 'train'
VAL_DIR    = BASE_DIR / 'val'
TEST_DIR   = BASE_DIR / 'test'
LABELS_CSV = BASE_DIR / 'labels.csv'

for sp in SPECIES:
    for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
        (split_dir / sp['code']).mkdir(parents=True, exist_ok=True)
    (RAW_DIR / sp['code']).mkdir(parents=True, exist_ok=True)

TARGET_PER_SPECIES = 260    # augmentation target per species -- kept below a high ceiling so the
                             # synthetic:real ratio stays sane (paired with MAX_AUG_RATIO below)
MAX_AUG_RATIO      = 1.5    # never let augmented images exceed 1.5x the real count for a species
IMAGES_PER_SPECIES = 400
MIN_RESOLUTION     = 224
BLUR_THRESHOLD     = 80.0
TRAIN_SPLIT        = 0.70
VAL_SPLIT          = 0.15
VIEW_TAGS          = ['full', 'bark', 'leaf', 'canopy', 'flower', 'fruit', 'root']

print('Config ready. Species:', [s['code'] for s in SPECIES])

Config ready. Species: ['TEK', 'NEM', 'MNG', 'PEP', 'BAM', 'MGR', 'RSW', 'MOR']


In [4]:
# -- Cell 4: iNaturalist download (unchanged -- API confirmed working in Cell 2) --
INAT_BASE = 'https://api.inaturalist.org/v1/observations'

def fetch_inat_urls(taxon_id, max_results=400):
    params = dict(taxon_id=taxon_id, quality_grade='research', photos='true',
                   per_page=min(max_results, 400), page=1, order_by='votes')
    try:
        r = requests.get(INAT_BASE, params=params, timeout=20)
        r.raise_for_status()
        results = r.json().get('results', [])
    except Exception as exc:
        print(f'  iNat API error: {exc}')
        return []
    items = []
    for obs in results:
        for photo in obs.get('photos', [])[:1]:
            url = photo.get('url', '').replace('/square.', '/medium.')
            if not url:
                continue
            view = 'full'
            for fv in obs.get('ofvs', []):
                val = str(fv.get('value', '')).lower()
                for tag in VIEW_TAGS:
                    if tag in val:
                        view = tag
                        break
            items.append(dict(url=url, view=view, obs_id=obs.get('id', 0),
                               license=photo.get('license_code', 'cc')))
    return items

def download_image(url, dest_path):
    try:
        r = requests.get(url, timeout=15, stream=True)
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        return True
    except Exception:
        return False

download_log = {}
for sp in SPECIES:
    code_ = sp['code']
    print(f'\nDownloading {sp["common"]} ({code_}) taxon_id={sp["inat_id"]}')
    urls = fetch_inat_urls(sp['inat_id'], max_results=IMAGES_PER_SPECIES)
    print(f'  Found {len(urls)} photo URLs')
    raw_paths = []
    for i, item in enumerate(tqdm(urls, desc=f'  {code_}', leave=False)):
        dest = RAW_DIR / code_ / f'raw_{i+1:04d}.jpg'
        if dest.exists():
            raw_paths.append((dest, item))
            continue
        if download_image(item['url'], dest):
            raw_paths.append((dest, item))
        time.sleep(0.3)
    download_log[code_] = raw_paths
    print(f'  Saved {len(raw_paths)} images')
print('\nAll species downloaded.')


  Found 200 photo URLs


  Saved 200 images

  Found 200 photo URLs


  Saved 200 images

  Found 200 photo URLs


  Saved 200 images

  Found 200 photo URLs


  Saved 200 images

  Found 200 photo URLs


  Saved 200 images

  Found 200 photo URLs


  Saved 200 images

  Found 156 photo URLs


  Saved 156 images

  Found 200 photo URLs


  Saved 200 images

All species downloaded.


In [5]:
# -- Cell 5: Validate images -- reject blurry, low-res, greyscale -------------
from collections import Counter

def laplacian_var(img_bgr):
    grey = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(grey, cv2.CV_64F).var()

def is_valid(path):
    try:
        img_pil = Image.open(path).convert('RGB')
        w, h = img_pil.size
    except Exception:
        return False, 'corrupt'
    if w < MIN_RESOLUTION or h < MIN_RESOLUTION:
        return False, f'low_res_{w}x{h}'
    img_bgr = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    if laplacian_var(img_bgr) < BLUR_THRESHOLD:
        return False, 'blurry'
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    if hsv[:, :, 1].mean() < 10:
        return False, 'greyscale'
    return True, 'ok'

valid_log, rejected_log = {}, {}
for sp in SPECIES:
    code_ = sp['code']
    ok_list, rej_list = [], []
    for (raw_path, meta) in download_log.get(code_, []):
        valid, reason = is_valid(raw_path)
        (ok_list if valid else rej_list).append((raw_path, meta) if valid else (raw_path, reason))
    valid_log[code_], rejected_log[code_] = ok_list, rej_list
    print(f'{code_}: PASS={len(ok_list)}  FAIL={len(rej_list)}')

TEK: PASS=200  FAIL=0
NEM: PASS=197  FAIL=3
MNG: PASS=198  FAIL=2
PEP: PASS=198  FAIL=2
BAM: PASS=200  FAIL=0
MGR: PASS=200  FAIL=0
RSW: PASS=155  FAIL=1
MOR: PASS=195  FAIL=5


In [6]:
# -- Cell 6: Rename -> {species}_{view}_{seq:04d}.jpg and split 70/15/15 ------
VIEW_MAP = {
    'full':'full','whole':'full','habit':'full','tree':'full','plant':'full',
    'bark':'bark','trunk':'bark','stem':'bark',
    'leaf':'leaf','leaves':'leaf','foliage':'leaf',
    'canopy':'canopy','crown':'canopy','top':'canopy',
    'flower':'flower','bloom':'flower',
    'fruit':'fruit','seed':'fruit','pod':'fruit',
    'root':'root',
}
def norm_view(raw):
    return VIEW_MAP.get(str(raw).lower(), 'full')

def split_list(items, tr=TRAIN_SPLIT, vl=VAL_SPLIT):
    items = items[:]
    random.shuffle(items)
    n = len(items)
    n_tr = math.ceil(n * tr)
    n_vl = math.ceil(n * vl)
    return items[:n_tr], items[n_tr:n_tr+n_vl], items[n_tr+n_vl:]

SPLIT_DIRS = {'train': TRAIN_DIR, 'val': VAL_DIR, 'test': TEST_DIR}
placed_files = {}

for sp in SPECIES:
    code_ = sp['code']
    name_lc = sp['common'].lower().replace(' ', '_')
    by_view = {}
    for item in valid_log.get(code_, []):
        v = norm_view(item[1].get('view', 'full'))
        by_view.setdefault(v, []).append(item)

    all_placed, seq = [], 1
    for view, group in by_view.items():
        tr, vl, te = split_list(group)
        for split_name, batch in [('train', tr), ('val', vl), ('test', te)]:
            for (raw_path, meta) in batch:
                fname = f'{name_lc}_{view}_{seq:04d}.jpg'
                dest = SPLIT_DIRS[split_name] / code_ / fname
                shutil.copy2(raw_path, dest)
                all_placed.append(dict(dest=dest, split=split_name, view=view, seq=seq,
                                        filename=fname, meta=meta, augmented=False))
                seq += 1
    placed_files[code_] = all_placed
    counts = {s: sum(1 for p in all_placed if p['split']==s) for s in ['train','val','test']}
    print(f'{code_}: total={len(all_placed)} {counts}')

TEK: total=200 {'train': 140, 'val': 30, 'test': 30}
NEM: total=197 {'train': 138, 'val': 30, 'test': 29}
MNG: total=198 {'train': 139, 'val': 30, 'test': 29}
PEP: total=198 {'train': 139, 'val': 30, 'test': 29}
BAM: total=200 {'train': 140, 'val': 30, 'test': 30}
MGR: total=200 {'train': 141, 'val': 30, 'test': 29}
RSW: total=155 {'train': 109, 'val': 24, 'test': 22}
MOR: total=195 {'train': 137, 'val': 30, 'test': 28}


In [7]:
# -- Cell 7: Capped augmentation -- never let synthetic images dominate -------
# Augmentation is capped at MAX_AUG_RATIO x the real image count, even if that leaves a species
# below TARGET_PER_SPECIES. Heavy reliance on augmented duplicates of a tiny real set is a direct
# overfitting driver -- better to train on a slightly smaller, more genuinely varied set than to
# pad it out with near-duplicate synthetic copies.
def augment_image(img, seed):
    random.seed(seed)
    img = img.copy()
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    img = img.rotate(random.uniform(-20, 20), expand=False)
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.25))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))
    w, h = img.size
    factor = random.uniform(0.88, 1.0)
    nw, nh = int(w*factor), int(h*factor)
    left, top = random.randint(0, w-nw), random.randint(0, h-nh)
    return img.crop((left, top, left+nw, top+nh)).resize((w, h), Image.BILINEAR)

for sp in SPECIES:
    code_ = sp['code']
    name_lc = sp['common'].lower().replace(' ', '_')
    train_imgs = [p for p in placed_files.get(code_, []) if p['split']=='train']
    n_train = len(train_imgs)
    max_aug = int(n_train * (MAX_AUG_RATIO - 1))
    needed  = min(max(0, TARGET_PER_SPECIES - n_train), max_aug)

    if needed == 0 or n_train == 0:
        print(f'{code_}: {n_train} train images -- no augmentation (cap reached or no real images)')
        continue

    print(f'{code_}: {n_train} train images, generating {needed} augmented (capped) ...')
    aug_seq, seed = 1, 0
    while aug_seq <= needed:
        src_item = train_imgs[(aug_seq-1) % len(train_imgs)]
        try:
            img = Image.open(src_item['dest']).convert('RGB')
        except Exception:
            aug_seq += 1
            continue
        fname = f"{name_lc}_{src_item['view']}_aug_{aug_seq:04d}.jpg"
        dest = TRAIN_DIR / code_ / fname
        augment_image(img, seed=seed).save(dest, quality=90)
        placed_files[code_].append(dict(dest=dest, split='train', view=src_item['view'],
                                         seq=src_item['seq'], filename=fname,
                                         meta=src_item['meta'], augmented=True))
        aug_seq += 1
        seed += 1
    final_train = sum(1 for p in placed_files[code_] if p['split']=='train')
    print(f'  -> {final_train} train images after capped augmentation')

TEK: 140 train images, generating 70 augmented (capped) ...
  -> 210 train images after capped augmentation
NEM: 138 train images, generating 69 augmented (capped) ...
  -> 207 train images after capped augmentation
MNG: 139 train images, generating 69 augmented (capped) ...
  -> 208 train images after capped augmentation
PEP: 139 train images, generating 69 augmented (capped) ...
  -> 208 train images after capped augmentation
BAM: 140 train images, generating 70 augmented (capped) ...
  -> 210 train images after capped augmentation
MGR: 141 train images, generating 70 augmented (capped) ...
  -> 211 train images after capped augmentation
RSW: 109 train images, generating 54 augmented (capped) ...
  -> 163 train images after capped augmentation
MOR: 137 train images, generating 68 augmented (capped) ...
  -> 205 train images after capped augmentation


## Note on `health_status` / quality / height / width

The xlsx (`labels_template` sheet) ships a `health_status` example column, and the
`species_reference` sheet has `dbh_cm` / `height_m` -- but those are **fixed per-species averages**,
not measurements taken from each individual photo. There is no real signal in the images for a model
to learn "this tree is 14m tall" or "this tree is stressed" -- those values are constant for every
photo of a given species. Training a head against them would just teach the network to memorize a
lookup table, dressed up as if it were reading the image. That's not a real capability, so it isn't
built here.

What this pipeline actually does for `co2_kg_yr` / `dbh_cm` / `height_m`: it predicts species (the one
thing the photos genuinely support), then looks the rest up from `species_lookup.json` (exported at
the end). That's the honest version of "detect height, width, quality" given this dataset.

If you want real height/width/condition predictions, you'd need labels that vary per photo -- e.g.
manually measured tree height per image, or a human-tagged health/condition per image.

In [8]:
# -- Cell 8: Write labels.csv ---------------------------------------------------
FIELDNAMES = ['image_id','filename','species_code','common_name','scientific_name',
              'co2_class','co2_kg_yr','dbh_cm','height_m','biome','view_type','source',
              'augmented','split']   # health_status column dropped -- see note above (no real labels)

LABELS_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(LABELS_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    for sp in SPECIES:
        code_ = sp['code']
        seq_by_view = {}
        for item in placed_files.get(code_, []):
            v = item['view']
            seq_by_view[v] = seq_by_view.get(v, 0) + 1
            image_id = f"{code_}-{v[:4].upper()}-{seq_by_view[v]:03d}"
            writer.writerow({
                'image_id': image_id, 'filename': item['filename'], 'species_code': code_,
                'common_name': sp['common'], 'scientific_name': sp['scientific'],
                'co2_class': sp['co2_class'], 'co2_kg_yr': sp['co2_kg_yr'],
                'dbh_cm': sp['dbh_cm'], 'height_m': sp['height_m'], 'biome': sp['biome'],
                'view_type': v.upper(), 'source': 'AUG' if item['augmented'] else 'iNAT',
                'augmented': 1 if item['augmented'] else 0, 'split': item['split'],
            })

import pandas as pd
df = pd.read_csv(LABELS_CSV)
print(f'labels.csv written -> {LABELS_CSV} ({len(df)} rows)')
print(df.groupby(['species_code','split']).size().unstack(fill_value=0).to_string())

labels.csv written -> /content/tree_dataset/labels.csv (2082 rows)
split         test  train  val
species_code                  
BAM             30    210   30
MGR             29    211   30
MNG             29    208   30
MOR             28    205   30
NEM             29    207   30
PEP             29    208   30
RSW             22    163   24
TEK             30    210   30


In [9]:
# -- Cell 9: GBIF cross-check -- resolved by NAME MATCH (see Cell 2), not by the xlsx key --
verified_map = {}
for sp in SPECIES:
    try:
        r = requests.get('https://api.gbif.org/v1/species/match',
                          params={'name': sp['scientific']}, timeout=15)
        r.raise_for_status()
        data = r.json()
        canonical = (data.get('canonicalName') or data.get('scientificName') or '').strip()
        ok = canonical.split()[:2] == sp['scientific'].split()[:2]
    except Exception as exc:
        print(f'  GBIF error for {sp["code"]}: {exc}')
        canonical, ok = 'error', False
    verified_map[sp['code']] = 1 if ok else 0
    print(f'  {sp["code"]:<4} {sp["scientific"]:<30} {"VERIFIED" if ok else "UNCONFIRMED"}  (GBIF: {canonical})')
    time.sleep(0.4)

n_verified = sum(verified_map.values())
print(f'\n{n_verified}/{len(SPECIES)} species verified against GBIF by name match.')
species_lookup = {sp['code']: {'common': sp['common'], 'scientific': sp['scientific'],
                                'co2_class': sp['co2_class'], 'co2_kg_yr': sp['co2_kg_yr'],
                                'dbh_cm': sp['dbh_cm'], 'height_m': sp['height_m'],
                                'biome': sp['biome'], 'gbif_verified': bool(verified_map[sp['code']])}
                   for sp in SPECIES}

  TEK  Tectona grandis                VERIFIED  (GBIF: Tectona grandis)
  NEM  Azadirachta indica             VERIFIED  (GBIF: Azadirachta indica)
  MNG  Mangifera indica               VERIFIED  (GBIF: Mangifera indica)
  PEP  Ficus religiosa                VERIFIED  (GBIF: Ficus religiosa)
  BAM  Bambusa vulgaris               VERIFIED  (GBIF: Bambusa vulgaris)
  MGR  Rhizophora mangle              VERIFIED  (GBIF: Rhizophora mangle)
  RSW  Dalbergia latifolia            VERIFIED  (GBIF: Dalbergia latifolia)
  MOR  Moringa oleifera               VERIFIED  (GBIF: Moringa oleifera)

8/8 species verified against GBIF by name match.


In [10]:
# -- Cell 10: Dataset summary --------------------------------------------------
df = pd.read_csv(LABELS_CSV)
summary = df.groupby(['species_code','split']).size().unstack(fill_value=0)
summary['aug'] = df[df['augmented']==1].groupby('species_code').size().reindex(summary.index, fill_value=0).astype(int)
summary['total'] = summary[['train','val','test']].sum(axis=1)
print(summary[['train','val','test','aug','total']].to_string())
print('\nDataset ready for training.')

split         train  val  test  aug  total
species_code                              
BAM             210   30    30   70    270
MGR             211   30    29   70    270
MNG             208   30    29   69    267
MOR             205   30    28   68    263
NEM             207   30    29   69    266
PEP             208   30    29   69    267
RSW             163   24    22   54    209
TEK             210   30    30   70    270

Dataset ready for training.


## EfficientNetB0 — regularization choices, and why

A small per-class image count (a few hundred per species) makes a transfer-learning CNN like this
prone to memorizing the training set rather than generalizing. Every choice below exists to push
back against that:
- **L2 weight decay** on both dense layers (1e-4) — discourages large weights that fit noise
- **Heavy dropout**: 0.5 / 0.3 — forces redundancy in the learned features instead of co-adapted units
- **Varied train-time augmentation** (flip, rotation, zoom, contrast, translation) — synthetic
  variation that's regenerated fresh each epoch, not baked into static files
- **Label smoothing** (0.1) in the loss — discourages overconfident predictions on a small dataset
- **Class weighting** — corrects for any imbalance between species with fewer real images
- **EarlyStopping monitors `val_loss`** (not `val_accuracy`) with `restore_best_weights=True` —
  val_accuracy can plateau noisily on a small val set while val_loss keeps climbing, which is the
  more reliable overfitting signal
- **ModelCheckpoint saves only the best `val_accuracy` epoch** — that's what gets exported, never
  whatever epoch training happened to stop on
- Fine-tuning only unfreezes the top 20 EfficientNet layers, at a very low learning rate, so the
  pretrained representation can't drift too far and overfit to a small dataset

In [11]:
# -- Cell 11: Build model --------------------------------------------------------
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE, BATCH_SIZE = 224, 32
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = sorted([sp['code'] for sp in SPECIES])
NUM_CLASSES = len(CLASS_NAMES)
class_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
print('Classes:', CLASS_NAMES, '->', NUM_CLASSES, 'output neurons')

def make_dataset(directory, shuffle=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory, labels='inferred', label_mode='categorical', class_names=CLASS_NAMES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=shuffle, seed=42)
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset(TRAIN_DIR, shuffle=True)
val_ds   = make_dataset(VAL_DIR)
test_ds  = make_dataset(TEST_DIR)
print(f'Train batches: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

# class weights -- corrects for species with fewer real images (e.g. RSW)
train_labels = []
for sp in SPECIES:
    n = sum(1 for p in placed_files[sp['code']] if p['split']=='train')
    train_labels += [class_idx[sp['code']]] * n
cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=np.array(train_labels))
class_weight = {i: w for i, w in enumerate(cw)}
print('Class weights:', {CLASS_NAMES[i]: round(w,2) for i,w in class_weight.items()})

aug_layer = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomTranslation(0.1, 0.1),
], name='aug')

L2 = regularizers.l2(1e-4)
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = aug_layer(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)
base.trainable = False

x = layers.GlobalAveragePooling2D()(base.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=L2)(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu', kernel_regularizer=L2)(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base.input, outputs=outputs)
model.summary(line_length=90)
print(f'Total params: {model.count_params():,}')

Classes: ['BAM', 'MGR', 'MNG', 'MOR', 'NEM', 'PEP', 'RSW', 'TEK'] -> 8 output neurons
Found 1622 files belonging to 8 classes.
Found 234 files belonging to 8 classes.
Found 226 files belonging to 8 classes.
Train batches: 51  Val: 8  Test: 8
Class weights: {'BAM': np.float64(0.97), 'MGR': np.float64(0.96), 'MNG': np.float64(0.97), 'MOR': np.float64(0.99), 'NEM': np.float64(0.98), 'PEP': np.float64(0.97), 'RSW': np.float64(1.24), 'TEK': np.float64(0.97)}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer) │ (None, 224, 224, 3)  │            0 │ -                     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ aug (Sequential)         │ (None, 224, 224, 3)  │            0 │ input_layer[0][0]     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ rescaling (Rescaling)    │ (None, 224, 224, 3)  │            0 │ aug[0][0]             │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ normalization            │ (None, 224, 224, 3)  │            7 │ rescaling[0][0]       │
│ (Normalization)          │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ rescaling_1 (Rescaling)  │ (None, 224, 224, 3)  │            0 │ normalization[0][0]   │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ stem_conv_pad            │ (None, 225, 225, 3)  │            0 │ rescaling_1[0][0]     │
│ (ZeroPadding2D)          │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ stem_conv (Conv2D)       │ (None, 112, 112, 32) │          864 │ stem_conv_pad[0][0]   │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ stem_bn                  │ (None, 112, 112, 32) │          128 │ stem_conv[0][0]       │
│ (BatchNormalization)     │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ stem_activation          │ (None, 112, 112, 32) │            0 │ stem_bn[0][0]         │
│ (Activation)             │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_dwconv           │ (None, 112, 112, 32) │          288 │ stem_activation[0][0] │
│ (DepthwiseConv2D)        │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_bn               │ (None, 112, 112, 32) │          128 │ block1a_dwconv[0][0]  │
│ (BatchNormalization)     │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_activation       │ (None, 112, 112, 32) │            0 │ block1a_bn[0][0]      │
│ (Activation)             │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_se_squeeze       │ (None, 32)           │            0 │ block1a_activation[0… │
│ (GlobalAveragePooling2D) │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_se_reshape       │ (None, 1, 1, 32)     │            0 │ block1a_se_squeeze[0… │
│ (Reshape)                │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_se_reduce        │ (None, 1, 1, 8)      │          264 │ block1a_se_reshape[0… │
│ (Conv2D)                 │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ block1a_se_expand        │ (None, 1, 1, 32)     │          288 │ block1a_se_reduce[0]

 Total params: 4,416,555 (16.85 MB)

 Trainable params: 364,424 (1.39 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Total params: 4,416,555


In [12]:
# -- Cell 12: Phase 1 -- train classification head only (base frozen) ---------
CKPT_PATH  = str(BASE_DIR / 'best_model.keras')
LOG_PATH_1 = str(BASE_DIR / 'training_log_phase1.csv')

callbacks_p1 = [
    ModelCheckpoint(CKPT_PATH, save_best_only=True, monitor='val_accuracy', verbose=1),
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss', verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-7, monitor='val_loss', verbose=1),
    CSVLogger(LOG_PATH_1, append=False),
]

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

print('Phase 1: training head only (base frozen)\n')
history1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                      callbacks=callbacks_p1, class_weight=class_weight, verbose=1)
print('\nPhase 1 done.')

Phase 1: training head only (base frozen)

Epoch 1/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.2090 - loss: 2.7308
Epoch 1: val_accuracy improved from None to 0.42735, saving model to /content/tree_dataset/best_model.keras

Epoch 1: finished saving model to /content/tree_dataset/best_model.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 28s 206ms/step - accuracy: 0.2546 - loss: 2.4733 - val_accuracy: 0.4274 - val_loss: 1.8571 - learning_rate: 0.0010
Epoch 2/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.3563 - loss: 1.9404
Epoch 2: val_accuracy improved from 0.42735 to 0.47436, saving model to /content/tree_dataset/best_model.keras

Epoch 2: finished saving model to /content/tree_dataset/best_model.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - accuracy: 0.3866 - loss: 1.8970 - val_accuracy: 0.4744 - val_loss: 1.7434 - learning_rate: 0.0010
Epoch 3/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.4210 - loss: 1.7804
Epoch 3: val_accuracy improved from 0.47436 to 0.508

In [13]:
# -- Cell 13: Phase 2 -- fine-tune top 20 EfficientNet layers (lower LR) ------
for layer in base.layers[-20:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

LOG_PATH_2 = str(BASE_DIR / 'training_log_phase2.csv')
callbacks_p2 = [
    ModelCheckpoint(CKPT_PATH, save_best_only=True, monitor='val_accuracy', verbose=1),
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss', verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-8, monitor='val_loss', verbose=1),
    CSVLogger(LOG_PATH_2, append=False),   # separate file from phase 1, so fine-tuning epochs don't overwrite head-training history
]

model.compile(optimizer=tf.keras.optimizers.Adam(5e-6),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

trainable = sum(w.numpy().size for w in model.trainable_weights)
print(f'Phase 2: fine-tuning top 20 EfficientNet layers. Trainable params: {trainable:,}\n')
history2 = model.fit(train_ds, validation_data=val_ds, epochs=20,
                      callbacks=callbacks_p2, class_weight=class_weight, verbose=1)
print('\nPhase 2 fine-tuning done.')

import pandas as pd
log1 = pd.read_csv(LOG_PATH_1); log1['phase'] = 1
log2 = pd.read_csv(LOG_PATH_2); log2['phase'] = 2
full_log = pd.concat([log1, log2], ignore_index=True)
full_log.to_csv(BASE_DIR / 'training_log.csv', index=False)
best_row = full_log.loc[full_log['val_accuracy'].idxmax()]
print(f"Best val_accuracy = {best_row['val_accuracy']*100:.2f}% "
      f"(phase {int(best_row['phase'])}, epoch {int(best_row['epoch'])}) -- this is the checkpoint that was saved.")

Phase 2: fine-tuning top 20 EfficientNet layers. Trainable params: 1,707,192

Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.6423 - loss: 1.3563
Epoch 1: val_accuracy improved from None to 0.60256, saving model to /content/tree_dataset/best_model.keras

Epoch 1: finished saving model to /content/tree_dataset/best_model.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 22s 186ms/step - accuracy: 0.6436 - loss: 1.3396 - val_accuracy: 0.6026 - val_loss: 1.4302 - learning_rate: 5.0000e-06
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.6438 - loss: 1.3431
Epoch 2: val_accuracy improved from 0.60256 to 0.60684, saving model to /content/tree_dataset/best_model.keras

Epoch 2: finished saving model to /content/tree_dataset/best_model.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 130ms/step - accuracy: 0.6504 - loss: 1.3203 - val_accuracy: 0.6068 - val_loss: 1.4273 - learning_rate: 5.0000e-06
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.6565 - loss: 1.3219
Epoch 3

In [14]:
# -- Cell 14: Evaluate best checkpoint on test set + plots ---------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

model.load_weights(CKPT_PATH)   # best val_accuracy checkpoint, not the last epoch
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f'\nTest accuracy : {test_acc*100:.2f}%')
print(f'Test loss     : {test_loss:.4f}')

y_true, y_pred = [], []
for imgs, labels_ in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels_.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
plt.title('Confusion Matrix - Test Set (best checkpoint)')
plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout()
plt.savefig(str(BASE_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

acc_all   = full_log['accuracy'].tolist()
vacc_all  = full_log['val_accuracy'].tolist()
loss_all  = full_log['loss'].tolist()
vloss_all = full_log['val_loss'].tolist()
ep = range(1, len(acc_all)+1)
p2_start = len(log1) + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ep, acc_all, label='Train'); axes[0].plot(ep, vacc_all, label='Val')
axes[0].axvline(p2_start, color='grey', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(ep, loss_all, label='Train'); axes[1].plot(ep, vloss_all, label='Val')
axes[1].axvline(p2_start, color='grey', linestyle='--')
axes[1].set_title('Loss'); axes[1].legend()
plt.suptitle('EfficientNetB0 - Training Curves')
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'training_curves.png'), dpi=150)
plt.show()

train_val_gap = max(acc_all) - max(vacc_all)
print(f'\nFinal train/val accuracy gap: {train_val_gap*100:.1f} pts. '
      f'A small gap here means the regularization is working; if it stays wide, the dataset likely '
      f'needs more real images per species rather than more training tricks.')

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.6460 - loss: 1.3293

Test accuracy : 64.60%
Test loss     : 1.3293

Classification Report:
              precision    recall  f1-score   support

         BAM       0.75      0.80      0.77        30
         MGR       0.77      0.93      0.84        29
         MNG       0.67      0.69      0.68        29
         MOR       0.58      0.39      0.47        28
         NEM       0.57      0.45      0.50        29
         PEP       0.58      0.72      0.65        29
         RSW       0.58      0.32      0.41        22
         TEK       0.59      0.77      0.67        30

    accuracy                           0.65       226
   macro avg       0.64      0.63      0.62       226
weighted avg       0.64      0.65      0.63       226


Final train/val accuracy gap: 8.9 pts. A small gap here means the regularization is working; if it stays wide, the dataset likely needs more real images per species rather than more training tricks.


## Quantization

Two TFLite exports are produced: a full-precision float32 model, and a full-integer INT8 model
(calibrated against a representative sample of the training set). INT8 is the lowest precision this
toolchain actually supports for a CNN of this kind, and is roughly 4x smaller than float32. A quick
agreement check against the float32 model's predictions is run below so you can see the quantized
model hasn't silently broken accuracy.

In [15]:
# -- Cell 15: Quantize (INT8 -- the real floor for this architecture) ----------
EXPORT_DIR = BASE_DIR / 'export'
EXPORT_DIR.mkdir(exist_ok=True)

# Float32 .tflite (baseline / max accuracy)
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32 = converter_fp32.convert()
(EXPORT_DIR / 'model_float32.tflite').write_bytes(tflite_fp32)
print(f'Float32 TFLite -> {len(tflite_fp32)/1e6:.1f} MB')

# Representative dataset for int8 calibration -- drawn from train set
def representative_dataset():
    for imgs, _ in train_ds.take(50):
        for img in imgs:
            yield [tf.expand_dims(img, axis=0)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()
(EXPORT_DIR / 'model_int8.tflite').write_bytes(tflite_int8)
print(f'INT8 TFLite    -> {len(tflite_int8)/1e6:.1f} MB '
      f'({len(tflite_fp32)/len(tflite_int8):.1f}x smaller than float32)')

# Sanity check: run the int8 model on a handful of val images and compare to the keras model
interpreter = tf.lite.Interpreter(model_content=tflite_int8)
interpreter.allocate_tensors()
in_detail, out_detail = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
matches, total = 0, 0
for imgs, labels_ in val_ds.take(3):
    fp_preds = np.argmax(model.predict(imgs, verbose=0), axis=1)
    for i in range(imgs.shape[0]):
        scale, zero_point = in_detail['quantization']
        img_q = (imgs[i:i+1].numpy() / scale + zero_point).astype(np.uint8) if scale else imgs[i:i+1].numpy().astype(np.uint8)
        interpreter.set_tensor(in_detail['index'], img_q)
        interpreter.invoke()
        out = interpreter.get_tensor(out_detail['index'])
        int8_pred = np.argmax(out, axis=1)[0]
        matches += int(int8_pred == fp_preds[i])
        total += 1
print(f'INT8 vs float32 agreement on {total} sampled val images: {matches}/{total}')

Saved artifact at '/tmp/tmp530c06p5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  133160798657168: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  133160798657360: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  133163911909392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911912272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911911504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911913040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911913616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911909200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911911312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133163911913424: TensorSpec(shape=(), dtype=tf.resource, name=None

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 TFLite    -> 5.3 MB (3.3x smaller than float32)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INT8 vs float32 agreement on 96 sampled val images: 66/96


In [16]:
# -- Cell 16: Export ONLY model artifacts + matrix images + logs -- download zip
keras_path = EXPORT_DIR / 'efficientnetb0_trees_best.keras'
model.save(str(keras_path))

with open(EXPORT_DIR / 'class_names.json', 'w') as f:
    json.dump({'class_names': CLASS_NAMES, 'class_to_idx': class_idx}, f, indent=2)

with open(EXPORT_DIR / 'species_lookup.json', 'w') as f:
    json.dump(species_lookup, f, indent=2)

# Copy supporting output files (not the dataset images) into the export folder
for fname in ['confusion_matrix.png', 'training_curves.png', 'training_log.csv',
              'training_log_phase1.csv', 'training_log_phase2.csv']:
    src = BASE_DIR / fname
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / fname)

with open(EXPORT_DIR / 'results_summary.txt', 'w') as f:
    f.write(f'Best val_accuracy : {best_row["val_accuracy"]*100:.2f}% '
            f'(phase {int(best_row["phase"])}, epoch {int(best_row["epoch"])})\n')
    f.write(f'Test accuracy     : {test_acc*100:.2f}%\n')
    f.write(f'Test loss         : {test_loss:.4f}\n')
    f.write(f'Train/val gap     : {train_val_gap*100:.1f} pts\n')
    f.write(f'INT8 vs FP32 agreement (sample) : {matches}/{total}\n')

import zipfile
zip_path = '/content/tree_model_export.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in EXPORT_DIR.rglob('*'):
        if fpath.is_file():
            zf.write(fpath, fpath.relative_to(EXPORT_DIR))

print('Files in download zip (model artifacts + plots + logs only -- no dataset images):')
for fpath in sorted(EXPORT_DIR.rglob('*')):
    if fpath.is_file():
        print(' ', fpath.relative_to(EXPORT_DIR), f'({fpath.stat().st_size/1e6:.2f} MB)')

from google.colab import files as colab_files
colab_files.download(zip_path)

Files in download zip (model artifacts + plots + logs only -- no dataset images):
  class_names.json (0.00 MB)
  confusion_matrix.png (0.06 MB)
  efficientnetb0_trees_best.keras (32.21 MB)
  model_float32.tflite (17.48 MB)
  model_int8.tflite (5.27 MB)
  results_summary.txt (0.00 MB)
  species_lookup.json (0.00 MB)
  training_curves.png (0.10 MB)
  training_log.csv (0.00 MB)
  training_log_phase1.csv (0.00 MB)
  training_log_phase2.csv (0.00 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>